# Welcome offers: free delivery or 10% off, profit per order

**Bayesian A/B Test for Two Means** · a worked example from the StatNex teaching guide

**The story.** A meal-kit shop tried two welcome offers for new customers: free delivery and 10% off. Chance decided which offer each customer got, 40 orders each. The shop recorded the profit on each order, after the cost of the offer.

**The question.** Which offer makes more profit per order, and is the difference worth switching for?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `pymc` and `arviz`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "pymc>=6.3" "arviz>=1.3"

## The table

This cell saves the table `14_bayesian/bayes_ab_means.csv` (40 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Free delivery,10% off
10.0,8.2
3.8,29.5
17.0,24.0
17.7,24.4
12.5,11.9
15.8,5.8
22.6,7.7
19.3,10.2
17.5,20.9
17.4,21.0
9.7,24.0
18.8,27.8
5.8,18.9
12.3,18.0
11.8,14.0
5.4,14.1
13.4,9.6
22.2,18.8
12.3,17.7
8.0,13.8
13.5,11.5
18.3,18.5
15.0,23.2
14.0,16.4
7.0,17.6
16.5,13.0
16.4,14.6
20.7,14.3
21.4,14.4
17.4,12.7
5.8,18.2
6.8,23.1
8.5,10.8
15.5,23.7
10.1,13.8
20.5,14.0
15.2,4.1
15.8,2.7
17.8,20.0
15.7,10.0

## Libraries

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
a_name, b_name = "Free delivery", "10% off"
credible = 95 / 100
low, high = (1 - credible) / 2, (1 + credible) / 2
margin = 1  # the smallest worthwhile difference


def observations(name):
    values = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
    return values


groups = [observations(a_name), observations(b_name)]

## Posteriors of the two means: PyMC

In [ ]:
# The model: each group's values are independent normal draws with the group's own mean
# and its own SD (the variances are not assumed equal, as in Welch's t test). The prior
# is the non-informative p(mu, sigma^2) ∝ 1/sigma^2 for each group: flat for mu and for
# log(sigma). PyMC samples the posterior by MCMC (NUTS), and the difference B - A is
# worked out draw by draw.
with pm.Model():
    mu = pm.Flat("mu", shape=2)
    log_sigma = pm.Flat("log_sigma", shape=2)  # flat in log(sigma): p(sigma^2) ∝ 1/sigma^2
    sigma = pm.Deterministic("sigma", pm.math.exp(log_sigma))
    for k, values in enumerate(groups):
        pm.Normal(f"y{k}", mu=mu[k], sigma=sigma[k], observed=values)
    pm.Deterministic("difference", mu[1] - mu[0])
    trace = pm.sample(draws=25_000, tune=2_000, chains=4, random_seed=2026, progressbar=False,
                      initvals={"mu": np.array([g.mean() for g in groups]),
                                "log_sigma": np.log([g.std(ddof=1) for g in groups])})
print(az.summary(trace, var_names=["mu", "sigma", "difference"], kind="stats", ci_prob=credible,
                 ci_kind="eti"))

means = trace.posterior["mu"].to_numpy().reshape(-1, 2)
difference = trace.posterior["difference"].to_numpy().ravel()
descriptive = pd.DataFrame({name: {"n": g.size, "mean": g.mean(), "sd": g.std(ddof=1)}
                            for name, g in zip((a_name, b_name), groups)})
# With 3 values or fewer in a group, its mean's posterior (a t with 2 df or fewer) has no SD.
posterior = pd.DataFrame({name: {"mean": means[:, k].mean(),
                                 "sd": means[:, k].std(ddof=1) if groups[k].size > 3 else np.nan,
                                 "lower": np.quantile(means[:, k], low), "upper": np.quantile(means[:, k], high)}
                          for k, name in enumerate((a_name, b_name))})
comparison = pd.DataFrame({"mean": [difference.mean()], "median": [np.median(difference)],
                           "lower": [np.quantile(difference, low)], "upper": [np.quantile(difference, high)]},
                          index=[f"Difference ({b_name} - {a_name})"])
# The decision: how often B beats A across the draws, and the expected loss of each
# choice -- the average over the draws of what it gives up: nothing in a draw where it
# is the better one, the other's lead where it is not. It is in the data's own units.
decision = {f"Probability that {b_name} beats {a_name}": np.mean(difference > 0),
            f"Probability that {a_name} beats {b_name}": np.mean(difference < 0)}
decision[f"Probability that {b_name} beats {a_name} by more than {margin:g}"] = np.mean(difference > margin)
decision[f"Expected loss if you choose {a_name}"] = np.mean(np.maximum(difference, 0))
decision[f"Expected loss if you choose {b_name}"] = np.mean(np.maximum(-difference, 0))
decision = pd.Series(decision)

print("\nDescriptive statistics")
print(descriptive)
print(f"\nPosterior of each population mean ({credible:.0%} credible interval)")
print(posterior)
print("\nPosterior of the difference")
print(comparison)
print("\nWhich to choose")
print(decision.to_string())

## What the result says

The mean profit per order is \$14.13 with free delivery and \$15.92 with 10% off. The posterior is what we believe about the true difference after seeing the data. Its mean is \$1.79 in favour of 10% off, but the 95% credible interval, the range that holds the true difference with 95% probability, runs from -0.79 to 4.37.

The probability that 10% off is the more profitable offer is 0.91. The probability that it wins by more than the \$1 that makes a switch worthwhile is only 0.73.

The expected loss is the profit a choice gives up on average: nothing when the choice is right, the shortfall when it is wrong. It is about \$0.05 an order for 10% off and \$1.85 for free delivery. So 10% off is probably the better choice, but the size of the gain is uncertain.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.